# 02 — Território: casos ou risco?

A incidência por 100 mil habitantes torna comparáveis populações de tamanhos diferentes.

In [ ]:
from pathlib import Path
import sys, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT))
from scripts.carregar_dados import caminhos
arq_casos, arq_municipios, FONTE = caminhos()
casos=pd.read_csv(arq_casos)
mun=pd.read_csv(arq_municipios,dtype={"codigo_ibge":"string"})
print("Fonte selecionada:",FONTE)
chave=["codigo_ibge"] if "codigo_ibge" in casos.columns and "codigo_ibge" in mun.columns else ["municipio"]
rank=casos.groupby(chave,as_index=False).casos.sum().merge(mun,on=chave)
rank["incidencia_100k"]=100000*rank.casos/rank.populacao
rank=rank.dropna(subset=["incidencia_100k"]).sort_values("incidencia_100k",ascending=False)
rank.head(12)

In [ ]:
top=rank.head(12).copy()
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=top.sort_values("casos",ascending=False),y="municipio",x="casos",ax=axes[0],color="#3182bd")
sns.barplot(data=top.sort_values("incidencia_100k",ascending=False),y="municipio",x="incidencia_100k",ax=axes[1],color="#e6550d")
axes[0].set_title("Contagem absoluta — 12 maiores incidências"); axes[1].set_title("Incidência por 100 mil")
for ax in axes: ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
import plotly.express as px
mapa=rank.dropna(subset=["latitude","longitude"]) if {"latitude","longitude"}.issubset(rank.columns) else rank.iloc[0:0]
if mapa.empty:
    print("Coordenadas ainda não disponíveis para a fonte selecionada; o ranking permanece válido.")
else:
    fig=px.scatter_map(mapa,lat="latitude",lon="longitude",size="incidencia_100k",color="incidencia_100k",hover_name="municipio",hover_data={"casos":True,"populacao":True},zoom=5.5,height=520,title=f"Intensidade territorial — {FONTE}")
    fig.show()

### Para discutir

Qual indicador deve orientar: comunicação pública, compra de insumos, equipes de campo e capacidade hospitalar? A resposta pode mudar conforme a decisão.